# Magnetospheric Convection: Coupling Functions & Convection Patterns
## Cowley (1982) — "The Causes of Convection in the Earth's Magnetosphere"

이 노트북은 Cowley (1982)에서 다룬 태양풍-자기권 결합 함수(coupling functions)와 대류 패턴을 구현합니다.

This notebook implements the solar wind-magnetosphere coupling functions and convection patterns discussed in Cowley (1982).

### 구현 내용 / Contents
1. **Part 1**: Coupling functions 구현 및 비교 / Implementation and comparison
2. **Part 2**: IMF 방향에 따른 대류 패턴 시각화 / Convection patterns vs IMF orientation
3. **Part 3**: Reconnection vs Viscous 전위차 분해 / Potential decomposition
4. **Part 4**: Burton equation과의 연결 / Connection to Burton equation
5. **Part 5**: Clock angle 의존성 분석 / Clock angle dependence analysis
6. **Part 6**: FTE vs Quasi-Steady reconnection 시뮬레이션 / FTE simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

RE = 6.371e6  # Earth radius [m]

## Part 1: Coupling Functions — 구현 및 비교 / Implementation and Comparison

Cowley가 평가한 주요 coupling functions:

1. **Half-wave rectifier** (Burton et al., 1975): $VB_s$ — 남향 IMF만 사용
2. **$\sin^2(\theta/2)$** (Kan & Lee, 1979): $VB_T\sin^2(\theta/2)$ — clock angle 의존
3. **Akasofu $\varepsilon$** (Akasofu, 1979): $VB^2\sin^4(\theta/2) \cdot l_0^2$ — 에너지 전달률
4. **Newell universal** (Newell et al., 2007): $V^{4/3}B_T^{2/3}\sin^{8/3}(\theta/2)$ — 현대적 개선

In [ ]:
# ============================================================
# Coupling function implementations (from scratch)
# ============================================================

def clock_angle(by: np.ndarray, bz: np.ndarray) -> np.ndarray:
    """IMF clock angle θ in the GSM Y-Z plane.

    Args:
        by: IMF By component [nT].
        bz: IMF Bz component [nT].

    Returns:
        theta: Clock angle [radians]. 0 = northward, π = southward.
    """
    return np.arctan2(np.abs(by), bz) % (2 * np.pi)


def bt_transverse(by: np.ndarray, bz: np.ndarray) -> np.ndarray:
    """Transverse IMF magnitude in the GSM Y-Z plane.

    Args:
        by: IMF By [nT].
        bz: IMF Bz [nT].

    Returns:
        BT: Transverse field magnitude [nT].
    """
    return np.sqrt(by**2 + bz**2)


def coupling_half_wave(v: np.ndarray, bz: np.ndarray) -> np.ndarray:
    """Burton et al. (1975) half-wave rectifier: V * Bs.

    Args:
        v: Solar wind speed [km/s].
        bz: IMF Bz [nT].

    Returns:
        Merging electric field proxy [mV/m].
    """
    bs = np.where(bz < 0, -bz, 0.0)  # Bs = |Bz| when southward, else 0
    return v * bs * 1e-3  # [km/s * nT → mV/m]


def coupling_sin2(v: np.ndarray, by: np.ndarray,
                  bz: np.ndarray) -> np.ndarray:
    """Kan & Lee (1979) merging electric field: V * BT * sin²(θ/2).

    Args:
        v: Solar wind speed [km/s].
        by: IMF By [nT].
        bz: IMF Bz [nT].

    Returns:
        Merging electric field [mV/m].
    """
    theta = clock_angle(by, bz)
    bt = bt_transverse(by, bz)
    return v * bt * np.sin(theta / 2)**2 * 1e-3


def coupling_epsilon(v: np.ndarray, by: np.ndarray,
                     bz: np.ndarray, l0_re: float = 7.0) -> np.ndarray:
    """Akasofu (1979) epsilon parameter: V * B² * sin⁴(θ/2) * l₀².

    Args:
        v: Solar wind speed [km/s].
        by: IMF By [nT].
        bz: IMF Bz [nT].
        l0_re: Effective length in Earth radii.

    Returns:
        Energy coupling rate [W].
    """
    theta = clock_angle(by, bz)
    b = np.sqrt(by**2 + bz**2)  # Total transverse B [nT]
    l0 = l0_re * RE  # [m]
    mu0 = 4 * np.pi * 1e-7
    # Convert: v [km/s→m/s], B [nT→T]
    return (v * 1e3) * (b * 1e-9)**2 * np.sin(theta / 2)**4 * l0**2 / mu0


def coupling_newell(v: np.ndarray, by: np.ndarray,
                    bz: np.ndarray) -> np.ndarray:
    """Newell et al. (2007) universal coupling function: V^(4/3) BT^(2/3) sin^(8/3)(θ/2).

    Args:
        v: Solar wind speed [km/s].
        by: IMF By [nT].
        bz: IMF Bz [nT].

    Returns:
        Coupling function [arbitrary units].
    """
    theta = clock_angle(by, bz)
    bt = bt_transverse(by, bz)
    return v**(4/3) * bt**(2/3) * np.sin(theta / 2)**(8/3)


print("Coupling functions defined:")
print("  1. coupling_half_wave(v, bz)      — Burton (1975)")
print("  2. coupling_sin2(v, by, bz)       — Kan & Lee (1979)")
print("  3. coupling_epsilon(v, by, bz)    — Akasofu (1979)")
print("  4. coupling_newell(v, by, bz)     — Newell (2007)")

## Part 2: IMF Clock Angle에 따른 Coupling Function 비교 / Coupling vs Clock Angle

IMF의 clock angle $\theta$를 0°(순수 북향)에서 360°(순수 북향)까지 변화시키며 각 coupling function의 응답을 비교합니다. Cowley는 half-wave rectifier가 $\sin^2(\theta/2)$의 특수한 경우(By=0)임을 지적했습니다.

Vary IMF clock angle from 0° (pure northward) to 360° and compare each coupling function's response. Cowley noted the half-wave rectifier is a special case of $\sin^2(\theta/2)$ when By=0.

In [ ]:
# ============================================================
# Coupling functions vs clock angle
# ============================================================

theta_deg = np.linspace(0, 360, 361)
theta_rad = np.deg2rad(theta_deg)
b_mag = 5.0  # nT
v_sw = 400.0  # km/s

# Decompose BT into By, Bz based on clock angle
by_arr = b_mag * np.sin(theta_rad)
bz_arr = b_mag * np.cos(theta_rad)

# Compute coupling functions (normalized to max=1 for shape comparison)
hw = coupling_half_wave(np.full_like(by_arr, v_sw), bz_arr)
s2 = coupling_sin2(np.full_like(by_arr, v_sw), by_arr, bz_arr)
eps = coupling_epsilon(np.full_like(by_arr, v_sw), by_arr, bz_arr)
nw = coupling_newell(np.full_like(by_arr, v_sw), by_arr, bz_arr)

# Normalize
hw_n = hw / np.max(hw) if np.max(hw) > 0 else hw
s2_n = s2 / np.max(s2) if np.max(s2) > 0 else s2
eps_n = eps / np.max(eps) if np.max(eps) > 0 else eps
nw_n = nw / np.max(nw) if np.max(nw) > 0 else nw

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Normalized coupling vs clock angle
ax = axes[0]
ax.plot(theta_deg, hw_n, "b-", linewidth=2, label="Half-wave rectifier (Burton)")
ax.plot(theta_deg, s2_n, "r-", linewidth=2, label="$\\sin^2(\\theta/2)$ (Kan & Lee)")
ax.plot(theta_deg, eps_n, "g--", linewidth=2, label="$\\sin^4(\\theta/2)$ (Akasofu ε)")
ax.plot(theta_deg, nw_n, "m:", linewidth=2, label="$\\sin^{8/3}(\\theta/2)$ (Newell)")
ax.set_xlabel("Clock angle θ [degrees]")
ax.set_ylabel("Normalized coupling")
ax.set_title("Coupling Function Shapes / 결합 함수 형태")
ax.legend(fontsize=9)
ax.set_xlim(0, 360)
ax.set_xticks([0, 45, 90, 135, 180, 225, 270, 315, 360])
ax.axvline(90, color="gray", linestyle=":", alpha=0.3)
ax.axvline(180, color="gray", linestyle=":", alpha=0.3)
ax.axvline(270, color="gray", linestyle=":", alpha=0.3)

# Annotate IMF directions
for deg, label in [(0, "N"), (90, "Dusk\n($B_y>0$)"), (180, "S"),
                   (270, "Dawn\n($B_y<0$)"), (360, "N")]:
    ax.text(deg, -0.08, label, ha="center", fontsize=9, color="gray")

# Right: Polar plot
ax = axes[1]
ax = plt.subplot(122, polar=True)
ax.plot(theta_rad, hw_n, "b-", linewidth=2, label="Half-wave")
ax.plot(theta_rad, s2_n, "r-", linewidth=2, label="$\\sin^2$")
ax.plot(theta_rad, eps_n, "g--", linewidth=2, label="$\\sin^4$")
ax.plot(theta_rad, nw_n, "m:", linewidth=2, label="$\\sin^{8/3}$")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.set_title("Polar View / 극좌표 표시", pad=20)
ax.legend(fontsize=8, loc="lower right", bbox_to_anchor=(1.3, 0))

plt.tight_layout()
plt.savefig("part2_coupling_vs_clock.png", dpi=150, bbox_inches="tight")
plt.show()

print("핵심 차이점 / Key differences:")
print("  - Half-wave: θ=90°에서 급격히 on/off (step function)")
print("  - sin²(θ/2): θ=90° 근처에서 점진적 전환 (smoother)")
print("  - sin⁴(θ/2): 남향 근처에 더 집중 (narrower peak)")
print("  - Newell sin^(8/3): sin²와 sin⁴의 중간")

## Part 3: Reconnection vs Viscous 전위차 분해 / Potential Decomposition

Cowley의 핵심 결론: $\Phi_{PC} = \Phi_R + \Phi_V$
- $\Phi_R$: Reconnection 기여 (IMF $B_z$에 강하게 의존)
- $\Phi_V$: Viscous 기여 (~20-30 kV, IMF에 약하게 의존)

Cowley's key conclusion: $\Phi_{PC} = \Phi_R + \Phi_V$

In [ ]:
# ============================================================
# Cross-polar cap potential: Reconnection + Viscous decomposition
# ============================================================

def cross_polar_cap_potential(v_sw: np.ndarray, by: np.ndarray,
                              bz: np.ndarray,
                              l_eff_re: float = 8.0,
                              phi_v: float = 25.0) -> dict:
    """Estimate cross-polar cap potential (Cowley framework).

    Args:
        v_sw: Solar wind speed [km/s].
        by: IMF By [nT].
        bz: IMF Bz [nT].
        l_eff_re: Effective reconnection X-line length [RE].
        phi_v: Viscous contribution [kV].

    Returns:
        Dict with 'phi_r', 'phi_v', 'phi_pc' in [kV].
    """
    em = coupling_sin2(v_sw, by, bz)  # [mV/m]
    l_eff = l_eff_re * RE  # [m]
    # Phi_R = Em * L_eff [mV/m * m = mV → kV: /1e6... wait]
    # Em in mV/m, L in m → Em*L in mV → /1e3 for kV
    phi_r = em * l_eff / 1e3  # [kV]
    phi_pc = phi_r + phi_v
    return {"phi_r": phi_r, "phi_v": phi_v, "phi_pc": phi_pc}


# Vary Bz from +10 to -20 nT, By=0, Vsw=400 km/s
bz_range = np.linspace(10, -20, 200)
by_zero = np.zeros_like(bz_range)
v_const = np.full_like(bz_range, 400.0)

pot = cross_polar_cap_potential(v_const, by_zero, bz_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Potential vs Bz
ax = axes[0]
ax.plot(bz_range, pot["phi_pc"], "b-", linewidth=2.5, label="$\\Phi_{PC}$ (total)")
ax.plot(bz_range, pot["phi_r"], "r--", linewidth=2, label="$\\Phi_R$ (reconnection)")
ax.axhline(pot["phi_v"], color="g", linestyle=":", linewidth=2,
           label=f"$\\Phi_V$ (viscous) = {pot['phi_v']:.0f} kV")
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)
ax.fill_between(bz_range, 0, pot["phi_v"], alpha=0.1, color="green")
ax.set_xlabel("IMF $B_z$ [nT]")
ax.set_ylabel("Cross-polar cap potential [kV]")
ax.set_title("$\\Phi_{PC}$ vs IMF $B_z$ (Cowley Framework)")
ax.legend(fontsize=10)
ax.set_xlim(10, -20)

# Annotate
ax.annotate("Northward\n(북향)", xy=(5, 30), fontsize=10, color="gray",
            ha="center")
ax.annotate("Southward\n(남향)", xy=(-15, 100), fontsize=10, color="blue",
            ha="center")

# Right: Fraction reconnection vs viscous
ax = axes[1]
frac_r = pot["phi_r"] / pot["phi_pc"] * 100
frac_v = pot["phi_v"] / pot["phi_pc"] * 100
ax.fill_between(bz_range, 0, frac_r, alpha=0.4, color="red",
                label="Reconnection %")
ax.fill_between(bz_range, frac_r, 100, alpha=0.4, color="green",
                label="Viscous %")
ax.set_xlabel("IMF $B_z$ [nT]")
ax.set_ylabel("Fraction of $\\Phi_{PC}$ [%]")
ax.set_title("Reconnection vs Viscous Contribution")
ax.legend(fontsize=10)
ax.set_xlim(10, -20)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig("part3_potential_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBz = -5 nT: Φ_R = {pot['phi_r'][bz_range < -4.9][0]:.1f} kV, "
      f"Φ_V = {pot['phi_v']:.0f} kV, "
      f"Φ_PC = {pot['phi_pc'][bz_range < -4.9][0]:.1f} kV")
print(f"Bz = -10 nT: Φ_R = {pot['phi_r'][bz_range < -9.9][0]:.1f} kV, "
      f"Φ_PC = {pot['phi_pc'][bz_range < -9.9][0]:.1f} kV")
print(f"\n결론: 남향 IMF에서 reconnection이 전위차의 대부분을 지배")

## Part 4: Burton Equation과의 연결 / Connection to Burton Equation

Cowley의 대류 프레임워크가 Burton (1975)의 half-wave rectifier를 어떻게 설명하는지 보여줍니다.
핵심: Burton의 injection function $Q(E_y)$는 reconnection-driven convection의 **직접적 결과**입니다.

Shows how Cowley's convection framework explains Burton's (1975) half-wave rectifier.
Key: Burton's injection function $Q(E_y)$ is a **direct consequence** of reconnection-driven convection.

In [ ]:
# ============================================================
# Burton equation with Cowley's sin²(θ/2) coupling
# Compare half-wave rectifier vs sin²(θ/2) in a storm scenario
# ============================================================

# Burton equation parameters
BURTON = {"a": 3.6e-5, "b": 15.8, "c": 20.0, "d": -1.5e-3, "Ec": 0.5}

def burton_with_coupling(v_sw, by, bz, n_sw, dt, dst0=0.0,
                          coupling="half_wave"):
    """Run Burton equation with different coupling functions.

    Args:
        coupling: "half_wave" or "sin2"
    """
    n = len(v_sw)
    mp = 1.6726e-27
    pdyn = 0.5 * mp * (n_sw * 1e6) * (v_sw * 1e3)**2 * 1e9

    if coupling == "half_wave":
        ey = -v_sw * bz * 1e-3  # Standard Ey = -V*Bz
        q = np.where(ey > BURTON["Ec"],
                     BURTON["d"] * (ey - BURTON["Ec"]), 0.0)
    elif coupling == "sin2":
        em = coupling_sin2(v_sw, by, bz)
        q = np.where(em > BURTON["Ec"],
                     BURTON["d"] * (em - BURTON["Ec"]), 0.0)

    dst_star = np.zeros(n)
    dst_star[0] = dst0 - BURTON["b"] * np.sqrt(pdyn[0]) + BURTON["c"]

    for i in range(n - 1):
        dst_star[i+1] = dst_star[i] + (q[i] - BURTON["a"] * dst_star[i]) * dt

    dst = dst_star + BURTON["b"] * np.sqrt(pdyn) - BURTON["c"]
    return dst, dst_star


# Storm scenario with varying IMF By component
dt = 150.0
hours = 48
n_steps = int(hours * 3600 / dt)
t_hr = np.arange(n_steps) * dt / 3600

v = np.full(n_steps, 450.0)
n_p = np.full(n_steps, 8.0)
bz_s = np.full(n_steps, 2.0)
by_s = np.full(n_steps, 0.0)

# Storm: Bz goes southward, By also significant (IMF rotation)
storm = (t_hr >= 6) & (t_hr < 18)
bz_s[storm] = -8.0
by_s[storm] = 6.0  # Significant By component!

# Recovery
rec = t_hr >= 18
bz_s[rec] = 2.0
by_s[rec] = 0.0

dst_hw, _ = burton_with_coupling(v, by_s, bz_s, n_p, dt, coupling="half_wave")
dst_s2, _ = burton_with_coupling(v, by_s, bz_s, n_p, dt, coupling="sin2")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[1, 2])

# Top: IMF components
ax = axes[0]
ax.plot(t_hr, bz_s, "b-", linewidth=1.5, label="$B_z$")
ax.plot(t_hr, by_s, "r--", linewidth=1.5, label="$B_y$")
bt = np.sqrt(by_s**2 + bz_s**2)
ax.plot(t_hr, bt, "k:", linewidth=1, label="$B_T$")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_ylabel("IMF [nT]")
ax.set_title("Burton Equation: Half-Wave vs sin²(θ/2) Coupling\n"
             "(Storm with significant IMF $B_y$)")
ax.legend(fontsize=10)
ax.set_xticklabels([])

# Bottom: Dst comparison
ax = axes[1]
ax.plot(t_hr, dst_hw, "b-", linewidth=2,
        label="Half-wave rectifier ($VB_s$, Burton 1975)")
ax.plot(t_hr, dst_s2, "r-", linewidth=2,
        label="$\\sin^2(\\theta/2)$ coupling (Cowley/Kan-Lee)")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Dst [nT]")
ax.legend(fontsize=10)

# Annotate difference
diff_idx = np.argmin(dst_s2)
ax.annotate(f"Δ = {dst_s2[diff_idx] - dst_hw[diff_idx]:.0f} nT\n"
            f"sin² gives deeper storm\nwhen $B_y$ is significant",
            xy=(t_hr[diff_idx], dst_s2[diff_idx]),
            xytext=(30, dst_s2[diff_idx] - 20),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=10, color="red")

plt.tight_layout()
plt.savefig("part4_burton_coupling.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Half-wave 최저 Dst: {dst_hw.min():.1f} nT")
print(f"sin²(θ/2) 최저 Dst: {dst_s2.min():.1f} nT")
print(f"차이: {dst_s2.min() - dst_hw.min():.1f} nT")
print(f"\nBy = {by_s[storm][0]} nT일 때 clock angle = "
      f"{np.rad2deg(clock_angle(by_s[storm][0:1], bz_s[storm][0:1]))[0]:.0f}°")
print("sin²(θ/2)가 By 성분의 추가 에너지를 포착하여 더 깊은 폭풍을 예측")

## Part 5: Ionospheric 2-Cell Convection Pattern / 전리층 2-Cell 대류 패턴

Cowley가 기술한 IMF 방향에 따른 전리층 대류 패턴을 간단한 모델로 시각화합니다.
Heppner-Maynard 스타일의 2-cell convection 패턴을 구현합니다.

Visualize ionospheric convection patterns as described by Cowley for different IMF orientations.
Implement Heppner-Maynard style 2-cell convection patterns.

In [ ]:
# ============================================================
# Simple 2-cell convection pattern model
# Based on a Volland-Stern type potential + viscous cells
# ============================================================

def convection_potential(mlat_deg, mlt_hr, phi_r=60.0, phi_v=25.0,
                         by_shift=0.0, bz_sign=-1):
    """Simple model of ionospheric convection potential.

    Args:
        mlat_deg: Magnetic latitude [degrees, 60-90].
        mlt_hr: Magnetic local time [hours, 0-24].
        phi_r: Reconnection potential [kV].
        phi_v: Viscous potential [kV].
        by_shift: Dawn-dusk asymmetry from IMF By [kV].
        bz_sign: -1 for southward, +1 for northward.

    Returns:
        Potential [kV].
    """
    colat = 90 - mlat_deg  # Colatitude [degrees]
    colat_rad = np.deg2rad(colat)
    mlt_rad = (mlt_hr / 24) * 2 * np.pi  # 0=midnight, π=noon

    # Reconnection-driven 2-cell (Volland-Stern like)
    if bz_sign < 0:  # Southward IMF
        # Main 2-cell: sin(MLT) pattern
        pot_r = phi_r * (colat_rad / np.deg2rad(30))**2 * np.sin(mlt_rad)
        # Clamp to polar cap region
        pot_r = np.where(colat < 30, pot_r, 0.0)
    else:  # Northward IMF — very weak, possibly reversed in polar cap
        pot_r = phi_r * 0.1 * (colat_rad / np.deg2rad(20))**2 * np.sin(mlt_rad)
        pot_r = np.where(colat < 20, pot_r, 0.0)

    # Viscous cells (narrow, on flanks)
    # Viscous cells are at dawn and dusk, tailward flow on flanks
    viscous_dawn = phi_v * np.exp(-((mlt_hr - 6)**2) / 4) * \
                   np.exp(-((colat - 15)**2) / 20)
    viscous_dusk = -phi_v * np.exp(-((mlt_hr - 18)**2) / 4) * \
                   np.exp(-((colat - 15)**2) / 20)
    pot_v = viscous_dawn + viscous_dusk

    # By asymmetry
    pot_by = by_shift * np.sin(mlt_rad) * (colat_rad / np.deg2rad(30))

    return pot_r + pot_v + pot_by


# Create polar grid
mlat = np.linspace(60, 90, 100)
mlt = np.linspace(0, 24, 200)
MLAT, MLT = np.meshgrid(mlat, mlt)

# Four IMF conditions (cf. Cowley's discussion)
conditions = [
    {"title": "Strong Southward\n$B_z=-10$, $B_y=0$",
     "phi_r": 80, "phi_v": 25, "by_shift": 0, "bz_sign": -1},
    {"title": "Southward + Dusk $B_y$\n$B_z=-5$, $B_y=+5$",
     "phi_r": 50, "phi_v": 25, "by_shift": 15, "bz_sign": -1},
    {"title": "Southward + Dawn $B_y$\n$B_z=-5$, $B_y=-5$",
     "phi_r": 50, "phi_v": 25, "by_shift": -15, "bz_sign": -1},
    {"title": "Northward\n$B_z=+5$, $B_y=0$",
     "phi_r": 10, "phi_v": 25, "by_shift": 0, "bz_sign": 1},
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12),
                          subplot_kw={"projection": "polar"})

for ax, cond in zip(axes.flat, conditions):
    POT = convection_potential(MLAT, MLT, **{k: v for k, v in cond.items()
                                              if k != "title"})

    # Convert to polar coordinates
    r = 90 - MLAT  # 0 at pole, 30 at 60° lat
    theta = (MLT / 24) * 2 * np.pi

    # Plot
    levels = np.linspace(-80, 80, 17)
    cs = ax.contourf(theta, r, POT, levels=levels, cmap="RdBu_r", alpha=0.7)
    ax.contour(theta, r, POT, levels=levels, colors="k", linewidths=0.5,
               alpha=0.5)

    ax.set_theta_zero_location("S")  # Noon at top (south in polar coords)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 30)
    ax.set_yticks([10, 20, 30])
    ax.set_yticklabels(["80°", "70°", "60°"])
    ax.set_xticklabels(["12", "", "18", "", "00", "", "06", ""])
    ax.set_title(cond["title"], pad=15, fontsize=10)

fig.suptitle("Ionospheric Convection Patterns (Cowley Framework)\n"
             "전리층 대류 패턴 (빨강=양, 파랑=음 전위)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("part5_convection_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

print("IMF 방향에 따른 대류 패턴 변화:")
print("  - 강한 남향: 대칭적 2-cell, 넓은 극관")
print("  - Bz<0 + By>0: dawn cell이 좁고 dusk cell이 넓음")
print("  - Bz<0 + By<0: dusk cell이 좁고 dawn cell이 넓음")
print("  - 북향: 매우 약한 대류, viscous cell만 남음")

## Part 6: FTE vs Quasi-Steady Reconnection / FTE 시뮬레이션

Cowley가 논의한 두 가지 reconnection 양식을 시뮬레이션합니다:
- **Quasi-steady**: 연속적이고 일정한 reconnection rate
- **Bursty (FTE-like)**: ~5분 간격의 간헐적 reconnection 펄스

두 양식이 동일한 총 flux transfer를 제공하더라도 시간적 프로파일이 어떻게 다른지 비교합니다.

Simulate the two reconnection modes Cowley discussed:
- **Quasi-steady**: continuous, constant reconnection rate
- **Bursty (FTE-like)**: intermittent pulses at ~5-min intervals

In [ ]:
# ============================================================
# FTE vs Quasi-Steady reconnection simulation
# ============================================================

dt_s = 10.0  # 10-second resolution
total_min = 60
n_steps = int(total_min * 60 / dt_s)
t_min = np.arange(n_steps) * dt_s / 60

# Reconnection rate (flux transfer rate) [Wb/s = V]
# Typical total: ~60 kV average reconnection potential

# Quasi-steady: constant rate
rate_steady = np.full(n_steps, 60e3)  # 60 kV continuous

# FTE-like: bursts every ~5 min, ~1 min duration, same total flux
fte_period_min = 5.0  # minutes between FTEs
fte_duration_min = 1.0  # duration of each FTE
duty_cycle = fte_duration_min / fte_period_min
peak_rate = 60e3 / duty_cycle  # Scale up so total flux matches

rate_fte = np.full(n_steps, 0.0)
for t_start in np.arange(2, total_min, fte_period_min):
    fte_mask = (t_min >= t_start) & (t_min < t_start + fte_duration_min)
    rate_fte[fte_mask] = peak_rate

# Also a "baseline + FTE" model (quasi-steady base + FTE bursts)
baseline = 30e3  # 30 kV baseline
fte_extra = 30e3 / duty_cycle  # Extra 30 kV from FTEs
rate_mixed = np.full(n_steps, baseline)
for t_start in np.arange(2, total_min, fte_period_min):
    fte_mask = (t_min >= t_start) & (t_min < t_start + fte_duration_min)
    rate_mixed[fte_mask] += fte_extra

# Cumulative flux transfer
flux_steady = np.cumsum(rate_steady * dt_s) / 1e6  # MWb
flux_fte = np.cumsum(rate_fte * dt_s) / 1e6
flux_mixed = np.cumsum(rate_mixed * dt_s) / 1e6

# Simulated Bn perturbation (bipolar signature of each FTE)
bn_fte = np.zeros(n_steps)
for t_start in np.arange(2, total_min, fte_period_min):
    t_center = t_start + fte_duration_min / 2
    # Bipolar: positive then negative
    bn_fte += 15 * np.exp(-((t_min - t_center + 0.2)**2) / 0.02) - \
              15 * np.exp(-((t_min - t_center - 0.2)**2) / 0.02)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), height_ratios=[1.5, 1, 1])

# Top: Reconnection rate
ax = axes[0]
ax.plot(t_min, rate_steady / 1e3, "b-", linewidth=1.5,
        label="Quasi-steady (60 kV)", alpha=0.8)
ax.plot(t_min, rate_fte / 1e3, "r-", linewidth=1.5,
        label=f"Pure FTE ({peak_rate/1e3:.0f} kV × {fte_duration_min:.0f} min / "
              f"{fte_period_min:.0f} min)", alpha=0.8)
ax.plot(t_min, rate_mixed / 1e3, "g-", linewidth=1.5,
        label="Mixed (30 kV base + FTE bursts)", alpha=0.8)
ax.set_ylabel("Reconnection\nrate [kV]")
ax.set_title("FTE vs Quasi-Steady Reconnection / FTE vs 준정상 재결합")
ax.legend(fontsize=9)
ax.set_xticklabels([])

# Middle: Cumulative flux
ax = axes[1]
ax.plot(t_min, flux_steady, "b-", linewidth=2, label="Quasi-steady")
ax.plot(t_min, flux_fte, "r--", linewidth=2, label="Pure FTE")
ax.plot(t_min, flux_mixed, "g:", linewidth=2, label="Mixed")
ax.set_ylabel("Cumulative flux\ntransfer [MWb]")
ax.legend(fontsize=9)
ax.set_xticklabels([])

# Bottom: Simulated Bn perturbation (FTE signature)
ax = axes[2]
ax.plot(t_min, bn_fte, "k-", linewidth=1)
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_ylabel("$B_n$ [nT]\n(magnetopause)")
ax.set_xlabel("Time [minutes]")
ax.set_title("Simulated FTE bipolar $B_n$ signature (cf. Figure 8)")

# Annotate one FTE
fte_idx = (t_min > 6.5) & (t_min < 8.5)
if np.any(fte_idx):
    ax.annotate("FTE: bipolar $B_n$\n(+/- perturbation)",
                xy=(7.2, bn_fte[t_min > 7.1][0]),
                xytext=(12, 12),
                arrowprops=dict(arrowstyle="->"),
                fontsize=10)

plt.tight_layout()
plt.savefig("part6_fte_simulation.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"총 플럭스 전달량 (60분간):")
print(f"  Quasi-steady: {flux_steady[-1]:.1f} MWb")
print(f"  Pure FTE:     {flux_fte[-1]:.1f} MWb")
print(f"  Mixed:        {flux_mixed[-1]:.1f} MWb")
print(f"\nCowley의 결론: 두 양식이 동일한 총 플럭스를 전달하더라도,")
print(f"FTE의 간헐적 특성이 자기권 응답에 영향을 줄 수 있다.")

## Summary / 요약

| Part | 구현 내용 / Content | 핵심 교훈 / Key Lesson |
|------|-------------------|----------------------|
| 1 | Coupling functions 구현 | 4가지 coupling function의 수학적 구조 이해 |
| 2 | Clock angle 의존성 비교 | Half-wave는 step function, sin²은 smooth transition |
| 3 | Reconnection vs Viscous 분해 | 남향 IMF에서 reconnection이 70-80% 지배 |
| 4 | Burton equation + sin²(θ/2) | By 성분이 클 때 sin²가 더 정확한 예측 제공 |
| 5 | 전리층 대류 패턴 | IMF By가 dawn-dusk 비대칭을 유발 |
| 6 | FTE 시뮬레이션 | Quasi-steady와 bursty가 동일한 총 flux를 전달 가능 |

### 다음 논문과의 연결 / Connection to Next Papers

- **#13 Richmond & Kamide (1988)**: AMIE로 Cowley의 대류 패턴을 실측 데이터에서 매핑
- **#14 Tsyganenko (1989)**: 대류에 의해 변형된 자기권 자기장의 경험적 모델
- **#15 Gonzalez et al. (1994)**: Reconnection → convection → storm의 인과 사슬 정리